In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import scipy.sparse

In [2]:
y_train = np.load('./data/y_train.npy')
y_test = np.load('./data/y_test.npy')

X_train = scipy.sparse.load_npz('./data/X_train.npz')
X_test = scipy.sparse.load_npz('./data/X_test.npz', )

In [3]:
import pytorch_lightning as pl
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.optim import RMSprop
from torch import nn, concat, Generator
from torchinfo import summary
from pytorch_lightning.utilities import rank_zero_only
from pytorch_lightning.callbacks import Callback
from torchmetrics import (MeanAbsoluteError,
                          R2Score,
                          Accuracy,
                          MeanSquaredError)

## Neural Network Model (2 layers)

### Class definitions

Some bits taken from Introduction to Statistical Learning

In [4]:
class BasicDataModule(pl.LightningDataModule):
    def __init__(self, X_train, y_train, X_test=None, y_test=None, batch_size=32, num_workers=0, validation=0.2, seed=0):
        super(BasicDataModule, self).__init__()

        self.batch_size = batch_size
        self.num_workers = num_workers

        full_dataset = TensorDataset(X_train, y_train)

        val_size = int(len(full_dataset) * validation) if isinstance(validation, float) else validation
        train_size = len(full_dataset) - val_size

        self.train_dataset, self.validation_dataset = random_split(
            full_dataset, [train_size, val_size]
        )

        if X_test is not None and y_test is not None:
            self.test_dataset = TensorDataset(X_test, y_test)
        else:
            self.test_dataset = None

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.validation_dataset, batch_size=self.batch_size, num_workers=self.num_workers)

    def test_dataloader(self):
        if self.test_dataset is None:
            return None
        return DataLoader(self.test_dataset, batch_size=self.batch_size, num_workers=self.num_workers)

In [5]:
class BasicModule(pl.LightningModule):
    def __init__(self, model, loss, optimizer=None, metrics=None, on_epoch=True, pre_process_y_for_metrics=lambda y: y):

        super(BasicModule, self).__init__()

        self.model = model
        self.loss = loss

        optimizer = optimizer or RMSprop(model.parameters())
        self._optimizer = optimizer
        self.metrics = metrics
        self.on_epoch = on_epoch
        self.pre_process_y_for_metrics = pre_process_y_for_metrics
        
    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        preds = self.forward(x)
        loss = self.loss(preds, y)
        self.log("train_loss", loss, on_epoch=self.on_epoch, on_step=False)

        y_ = self.pre_process_y_for_metrics(y)
        for _metric in self.metrics.keys():
            pl_metric = self.metrics[_metric]
            self.log(f"train_{_metric}", pl_metric(preds.to(pl_metric.device), y_.to(pl_metric.device)), on_epoch=self.on_epoch)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch

    @rank_zero_only
    def validation_step(self, batch, batch_idx):
        x, y = batch

    def predict_step(self, batch, batch_idx):
        x, y = batch
        return y, self.forward(x)

    def configure_optimizers(self):
        return self._optimizer

    @staticmethod
    def regression(model, metrics=None, device='cpu', **kwargs):

        if metrics is None:
            metrics = {}

        loss = nn.MSELoss().to(device)
        if device is not None:
            for key, metric in metrics.items():
                metrics[key] = metric.to(device)
        return BasicModule(model, loss, metrics=metrics, **kwargs)

    @staticmethod
    def binary_classification(model, metrics=None, device='cpu', **kwargs):

        if metrics is None:
            metrics = {}

        loss = nn.BCEWithLogitsLoss()
        if 'accuracy' not in metrics:
            metrics['accuracy'] = Accuracy('binary')
        if device is not None:
            for key, metric in metrics.items():
                metrics[key] = metric.to(device)
        return BasicModule(model, loss, metrics=metrics, pre_process_y_for_metrics = lambda x: x.int(), **kwargs)

    @staticmethod
    def classification(model, num_classes, metrics=None, device='cpu', **kwargs):
        if metrics is None:
            metrics = {}
        loss = nn.CrossEntropyLoss().to(device)
        if 'accuracy' not in metrics:
            metrics['accuracy'] = Accuracy('multiclass', num_classes=num_classes)
        if device is not None:
            for key, metric in metrics.items():
                metrics[key] = metric.to(device)

        return BasicModule(model, loss, metrics=metrics, **kwargs)

In [6]:
class BasicErrorTracker(Callback):

    def on_validation_epoch_start(self, trainer, pl_module):
        self.val_preds = []
        self.val_targets = []

    def on_validation_batch_start(self, trainer, pl_module, batch, batch_idx, dataloader_idx=0):
        x, y = batch
        self.val_preds.append(pl_module.forward(x))
        self.val_targets.append(y)

    def on_validation_epoch_end(self, trainer, pl_module):
        preds = concat(self.val_preds)
        targets = concat(self.val_targets)
        targets_ = pl_module.pre_process_y_for_metrics(targets)

        loss = pl_module.loss(preds, targets)
        pl_module.log("valid_loss", loss, on_epoch=pl_module.on_epoch)

        for _metric in pl_module.metrics.keys():
            pl_metric = pl_module.metrics[_metric]
            pl_module.log(f"valid_{_metric}",
                          pl_metric(preds.to(pl_metric.device), targets_.to(pl_metric.device)),
                          on_epoch=pl_module.on_epoch)

    def on_test_epoch_start(self, trainer, pl_module):
        self.test_preds = []
        self.test_targets = []

    def on_test_batch_start(self, trainer, pl_module, batch, batch_idx, dataloader_idx=0):
        x, y = batch
        self.test_preds.append(pl_module.forward(x))
        self.test_targets.append(y)

    def on_test_epoch_end(self, trainer, pl_module):
        preds = concat(self.test_preds)
        targets = concat(self.test_targets)
        targets_ = pl_module.pre_process_y_for_metrics(targets)
        
        loss = pl_module.loss(preds, targets)
        pl_module.log("test_loss", loss, on_epoch=pl_module.on_epoch)

        for _metric in pl_module.metrics.keys():
            pl_metric = pl_module.metrics[_metric]
            pl_module.log(f"test_{_metric}",
                          pl_metric(preds.to(pl_metric.device), targets_.to(pl_metric.device)),
                          on_epoch=pl_module.on_epoch)

### Convert data to tensor

Note: Memory intensive for large files!

In [7]:
X_train_dense = X_train.toarray() # to numpy format
X_train_tensor = torch.from_numpy(X_train_dense).float() # To tensor format
y_train_tensor = torch.from_numpy(y_train).long()

X_test_dense = X_test.toarray() # to numpy format
X_test_tensor = torch.from_numpy(X_test_dense).float() # To tensor format
y_test_tensor = torch.from_numpy(y_test).long()

del X_train_dense, X_test_dense

In [8]:
nn_dm = BasicDataModule(
    X_train=X_train_tensor,
    y_train=y_train_tensor,
    X_test=X_test_tensor,
    y_test=y_test_tensor,
    validation=0.2, num_workers=6, batch_size=512)

### Model definition

In [9]:
class nn2L(nn.Module):

    def __init__(self, input_size):
        super(nn2L, self).__init__()
        self.dense1 = nn.Linear(input_size, 16)
        self.activation = nn.ReLU()
        self.dense2 = nn.Linear(16, 16)
        self.output = nn.Linear(16, 5)

    def forward(self, x):
        val = self.activation(self.dense1(x))
        val = self.activation(self.dense2(val))
        return self.output(val)

In [10]:
# X_train_tensor.size()[1] is input size
nn_model = nn2L(X_train_tensor.size()[1])

Model summary of parameters to estimate

In [11]:
summary(nn_model, input_size=X_train_tensor.size(), col_names=['input_size', 'output_size', 'num_params'])

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
nn2L                                     [187896, 5000]            [187896, 5]               --
├─Linear: 1-1                            [187896, 5000]            [187896, 16]              80,016
├─ReLU: 1-2                              [187896, 16]              [187896, 16]              --
├─Linear: 1-3                            [187896, 16]              [187896, 16]              272
├─ReLU: 1-4                              [187896, 16]              [187896, 16]              --
├─Linear: 1-5                            [187896, 16]              [187896, 5]               85
Total params: 80,373
Trainable params: 80,373
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 15.10
Input size (MB): 3757.92
Forward/backward pass size (MB): 55.62
Params size (MB): 0.32
Estimated Total Size (MB): 3813.86

### Training

In [12]:
nn_optimizer = RMSprop(nn_model.parameters(), lr=0.001)
nn_module = BasicModule.classification(model=nn_model, num_classes=5, optimizer=nn_optimizer)

In [13]:
nn_logger = pl.loggers.CSVLogger('logs', name='nn2L')
nn_trainer = pl.Trainer(deterministic=True, logger=nn_logger, max_epochs=20, callbacks=[BasicErrorTracker()]) # log_every_n_steps=45
nn_trainer.fit(nn_module, datamodule=nn_dm)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name  | Type             | Params | Mode 
---------------------------------------------------
0 | model | nn2L             | 80.4 K | train
1 | loss  | CrossEntropyLoss | 0      | train
---------------------------------------------------
80.4 K    Trainable params
0         Non-trainable params
80.4 K    Total params
0.321     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/andy/anaconda3/envs/erdos/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:420: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.
/Users/andy/anaconda3/envs/erdos/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:420: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.


In [14]:
test_results = nn_trainer.test(nn_module, datamodule=nn_dm)

/Users/andy/anaconda3/envs/erdos/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:420: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.6542203426361084
        test_loss           0.9166735410690308
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


In [15]:
test_results

[{'test_loss': 0.9166735410690308, 'test_accuracy': 0.6542203426361084}]